In [ ]:
# ==============================================================================
# HÜCRE 0: FL ALTYAPISI, NUMPY DÜZELTMESİ VE OTOMATİK YENİDEN BAŞLATMA
# ==============================================================================
import os
import time

print("1. Protobuf çakışması önleniyor...")
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

print("2. Federe Öğrenme motoru (Flower & Ray) ve Timm sessizce kuruluyor...")
!pip install -q flwr[simulation] ray timm "protobuf==3.20.3"

print("3. PyTorch çökmesini engellemek için NumPy (96 byte) ZORLA güncelleniyor...")
!pip install -q -U numpy

print("\n✅ BÜTÜN KURULUMLAR BAŞARILI!")
print("Değişikliklerin belleğe kazınması için sistem 3 saniye içinde OTOMATİK olarak yeniden başlatılacak...")
print("DİKKAT: Ekranın sol aslında 'Oturum çöktü' veya 'Yeniden başlatılıyor' uyarısı alırsanız PANİK YAPMAYIN, bu işlemi biz tetikledik!")

time.sleep(3) # Mesajı okuyabilmeniz için kısa bir bekleme
os.kill(os.getpid(), 9) # Çekirdeği vurur ve Colab'i anında yeniden başlatmaya zorlar

1. Protobuf çakışması önleniyor...
2. Federe Öğrenme motoru (Flower & Ray) ve Timm sessizce kuruluyor...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.2/157.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 141.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [1]:
# ==============================================================================
# HÜCRE 1: KÜTÜPHANE İÇE AKTARIMLARI VE VERİ ORTAMININ HAZIRLANMASI
# ==============================================================================

# ==============================================================================
# TEMEL KÜTÜPHANELERİN İÇE AKTARILMASI
# ==============================================================================
import os           # İşletim sistemi işlemleri, klasör/dosya yolu yönetimi
import time         # Eğitim sürelerinin ve çıkarım (inference) hızlarının hesaplanması
import zipfile      # Sıkıştırılmış veri setinin (MURA) yerel diske çıkartılması
import shutil       # Dosyaların yerel diske güvenli kopyalanması için

# ==============================================================================
# PYTORCH VE DERİN ÖĞRENME MODÜLLERİ
# ==============================================================================
import torch        # Ana derin öğrenme çerçevesi
import torch.nn as nn # Sinir ağı katmanları ve kayıp (loss) fonksiyonları
import torch.optim as optim # Optimizasyon algoritmaları (AdamW vb. ve Öğrenme Oranı Planlayıcıları)
from torch.utils.data import Dataset, DataLoader # Özelleştirilmiş veri seti ve toplu veri yükleyici sınıfları
from torchvision import transforms, models # Dinamik veri artırımı (augmentation) ve ön-eğitimli jenerik modeller

# ==============================================================================
# VERİ İŞLEME VE GÖRSELLEŞTİRME MODÜLLERİ
# ==============================================================================
from PIL import Image # Tıbbi görüntülerin (Röntgen) diskten okunması
import numpy as np  # Vektörel matris işlemleri
import pandas as pd # Eğitim geçmişinin ve sonuçların tablo halinde kaydedilmesi (CSV)
from tqdm import tqdm # Eğitim ve doğrulama döngülerinde iterasyon ilerleyişinin görselleştirilmesi

# ==============================================================================
# PERFORMANS METRİKLERİ (SCIKIT-LEARN)
# ==============================================================================
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, cohen_kappa_score, confusion_matrix,
                             roc_auc_score, average_precision_score,
                             mean_absolute_error, mean_squared_error)

# ==============================================================================
# ÇALIŞMA ORTAMI VE VERİ BAZI HAZIRLIKLARI
# ==============================================================================
# 1. Google Drive Bağlantısı: Ağırlıkların ve grafiklerin kaydedileceği kalıcı disk
from google.colab import drive
drive.mount('/content/drive')

# 2. Veri Setinin Yerel Diske Çıkartılması (Kesintisiz İndirme Stratejisi)
# Bulut depolamadan (Drive) doğrudan veri okumak I/O darboğazına (bottleneck) ve kopmalara yol açar.
# Ağ zaman aşımı (Errno 107) ve eksik veri aktarımını önlemek için veri seti
# önce geçici yerel belleğe kopyalanır, ardından çıkartılır.
DRIVE_ZIP_YOLU = '/content/drive/MyDrive/Doktora/Verisetleri_tik/Islenmis_MURA.zip'
YEREL_ZIP_YOLU = '/content/Islenmis_MURA_Temp.zip'
YEREL_VERI_KLASORU = '/content/dataset'

if not os.path.exists(YEREL_VERI_KLASORU):
    print("1. Ağ kopmalarını engellemek için ZIP dosyası yerel diske kopyalanıyor (Lütfen bekleyin)...")
    shutil.copy(DRIVE_ZIP_YOLU, YEREL_ZIP_YOLU)

    print("2. Kopyalanan ZIP dosyası yerel klasöre çıkartılıyor...")
    with zipfile.ZipFile(YEREL_ZIP_YOLU, 'r') as zip_ref:
        zip_ref.extractall(YEREL_VERI_KLASORU)

    print("3. Geçici ZIP dosyası temizleniyor...")
    os.remove(YEREL_ZIP_YOLU)
    print("Veri seti eksiksiz ve güvenli bir şekilde hazırlandı.")
else:
    print("Veri seti yerel diskte zaten mevcut.")

# ==============================================================================
# DONANIM HIZLANDIRICI ATAMASI
# ==============================================================================
# A100/T4 GPU Kontrolü: Tensor işlemlerinin CPU yerine CUDA mimarisinde yürütülmesi için
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Aktif İşlem Birimi: {device}")

Mounted at /content/drive
1. Ağ kopmalarını engellemek için ZIP dosyası yerel diske kopyalanıyor (Lütfen bekleyin)...
2. Kopyalanan ZIP dosyası yerel klasöre çıkartılıyor...
3. Geçici ZIP dosyası temizleniyor...
Veri seti eksiksiz ve güvenli bir şekilde hazırlandı.
Aktif İşlem Birimi: cuda


hücre 1 sözde kod

In [4]:
# ==============================================================================
# HÜCRE 2: FEDERE ÖĞRENME HİPERPARAMETRELERİ
# ==============================================================================
CONFIG = {
    "experiment_name": "Exp 6.1.1.2: MobileViT-S_FL_IID_LocalEpoch:5_Round:10",
    "model_architecture": "mobilevit_s",
    "freeze_backbone": False,

    # --- MERKEZİ EĞİTİMDEN AKTARILAN SABİT PARAMETRELER ---
    "target_image_size": (224, 224),
    "batch_size": 32,
    "learning_rate": 5e-5,
    "weight_decay": 5e-3,
    "patience": 12,           # early_stopping_patience
    "num_workers": 4,
    "use_mixup": False,
    "mixup_alpha": 0.2,

    # --- FL STRATEJİSİ VE ABLASYON AYARLARI ---
    "num_clients": 5,
    "fraction_fit": 1.0,
    "lr_decay": 0.95,

    # Deney 6.1.1.2 Özel Ayarları (Toplam 50 Epok Sabit)
    "num_rounds": 10,
    "local_epochs": 5,

    "augmentations": {
        "random_rotation_degrees": 15,
        "horizontal_flip_prob": 0.5,
        "color_jitter_brightness": 0.2,
        "color_jitter_contrast": 0.2
    }
}
print(f"✅ {CONFIG['experiment_name']} konfigürasyonu başarıyla yüklendi.")

✅ Exp 6.1.1.2: MobileViT-S_FL_IID_LocalEpoch:5_Round:10 konfigürasyonu başarıyla yüklendi.


hücre 2 sözde kod

In [5]:
# ==============================================================================
# HÜCRE 3: ÖZEL MURA VERİ SETİ SINIFI VE IID PARÇALAMA (KESİN ÇÖZÜM)
# ==============================================================================
import os
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

YEREL_VERI_KLASORU = '/content/dataset'

if os.path.exists(os.path.join(YEREL_VERI_KLASORU, 'train')):
    TRAIN_DIR = os.path.join(YEREL_VERI_KLASORU, 'train')
    VALID_DIR = os.path.join(YEREL_VERI_KLASORU, 'valid')
else:
    TRAIN_DIR = os.path.join(YEREL_VERI_KLASORU, 'MURA-v1.1', 'train')
    VALID_DIR = os.path.join(YEREL_VERI_KLASORU, 'MURA-v1.1', 'valid')

print("MURA Veri Seti ikili sınıflandırma (0: Sağlam, 1: Kırık) için taranıyor...")

# 1. ÖZEL MURA VERİ OKUYUCU SINIFI (Custom Dataset)
class MuraBinaryDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        # Tüm alt klasörleri (XR_ELBOW vb.) gez ve resimleri topla
        for root, _, files in os.walk(root_dir):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    full_path = os.path.join(root, file)
                    self.image_paths.append(full_path)

                    # MURA Veri Seti Mantığı: Klasör veya dosya adında 'positive' veya '1' varsa kırık (1), yoksa sağlam (0)
                    # Not: Merkezi eğitimde kullandığınız klasörleme yapısına göre Kırık etiketini 1 yapıyoruz.
                    if 'positive' in full_path.lower() or '/1/' in full_path.lower() or '_1.' in full_path.lower():
                        self.labels.append(1) # Kırık / Anormal
                    else:
                        self.labels.append(0) # Sağlam / Normal

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# 2. Transformasyonlar
train_transforms = transforms.Compose([
    transforms.Resize(CONFIG["target_image_size"]),
    transforms.RandomRotation(CONFIG["augmentations"]["random_rotation_degrees"]),
    transforms.RandomHorizontalFlip(p=CONFIG["augmentations"]["horizontal_flip_prob"]),
    transforms.ColorJitter(brightness=CONFIG["augmentations"]["color_jitter_brightness"], contrast=CONFIG["augmentations"]["color_jitter_contrast"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

valid_transforms = transforms.Compose([
    transforms.Resize(CONFIG["target_image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Veri Setlerini Yükleme (Artık ImageFolder değil, kendi sınıfımız)
full_train_dataset = MuraBinaryDataset(root_dir=TRAIN_DIR, transform=train_transforms)
global_valid_dataset = MuraBinaryDataset(root_dir=VALID_DIR, transform=valid_transforms)

# Etiket Dağılımını Kontrol Edelim (Jüri için önemli bir istatistik)
train_labels = full_train_dataset.labels
print(f"🔥 Sınıf Dağılımı: 0 (Sağlam): {train_labels.count(0)} adet | 1 (Kırık): {train_labels.count(1)} adet 🔥")

# 4. Sunucu DataLoader'ı
global_val_loader = DataLoader(global_valid_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])

# 5. İstemciler İçin IID Parçalama
num_clients = CONFIG["num_clients"]
total_train_size = len(full_train_dataset)

split_size = total_train_size // num_clients
lengths = [split_size] * num_clients
lengths[-1] += total_train_size % num_clients

client_datasets = random_split(full_train_dataset, lengths, generator=torch.Generator().manual_seed(42))

def get_client_dataloader(cid: int):
    client_data = client_datasets[cid]
    return DataLoader(client_data, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"])

print("\n[BAŞARILI] Veri Seti IID Parçalama Tamamlandı!")

MURA Veri Seti ikili sınıflandırma (0: Sağlam, 1: Kırık) için taranıyor...
🔥 Sınıf Dağılımı: 0 (Sağlam): 21935 adet | 1 (Kırık): 14873 adet 🔥

[BAŞARILI] Veri Seti IID Parçalama Tamamlandı!


hücre 3 sözde kod

In [6]:
# ==============================================================================
# HÜCRE 4: FL İSTEMCİ (CLIENT) MİMARİSİ (DİNAMİK LR DESTEKLİ)
# ==============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from collections import OrderedDict
import flwr as fl

def jenerik_model_olustur(model_adi, num_classes=2, dropout_rate=0.5):
    # (Öncekiyle aynı model oluşturma kodları...)
    if model_adi == "mobilevit_s":
        import timm
        model = timm.create_model("mobilevit_s", pretrained=True, num_classes=num_classes)
    elif model_adi == "mobilenet_v3_large":
        model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        model.classifier[3] = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(model.classifier[3].in_features, num_classes))
    else:
        raise ValueError(f"HATA: '{model_adi}' tanımlı değil.")

    freeze_backbone = CONFIG.get("freeze_backbone", False)
    for name, param in model.named_parameters():
        if not freeze_backbone:
            param.requires_grad = True
        else:
            if any(keyword in name for keyword in ["classifier", "fc", "head"]):
                param.requires_grad = True
            else:
                param.requires_grad = False
    return model

# DİKKAT: Artık sabit CONFIG["learning_rate"] yerine current_lr alıyor!
def train(model, trainloader, epochs, device, current_lr):
    criterion = nn.CrossEntropyLoss()
    head_params, backbone_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad: continue
        if any(keyword in name for keyword in ["fc", "classifier", "head"]):
            head_params.append(param)
        else:
            backbone_params.append(param)

    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': current_lr / 10},
        {'params': head_params, 'lr': current_lr}
    ], weight_decay=CONFIG["weight_decay"])

    model.train()
    for epoch in range(epochs):
        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

def test(model, testloader, device):
    criterion = nn.CrossEntropyLoss()
    correct, total, loss = 0, 0, 0.0
    model.eval()
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return loss / len(testloader), correct / total

class MuraClient(fl.client.NumPyClient):
    def __init__(self, cid, model, trainloader, valloader, device):
        self.cid = cid
        self.model = model
        self.trainloader = trainloader
        self.valloader = valloader
        self.device = device

    def get_parameters(self, config):
        return [val.cpu().numpy() for _, val in self.model.state_dict().items()]

    def set_parameters(self, parameters):
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        # Sunucunun hesapladığı bu turdaki güncel LR'yi çekiyoruz
        current_lr = config.get("lr", CONFIG["learning_rate"])
        train(self.model, self.trainloader, epochs=CONFIG["local_epochs"], device=self.device, current_lr=current_lr)
        return self.get_parameters(config={}), len(self.trainloader.dataset), {}

print("[BAŞARILI] İstemci Altyapısı Dinamik LR Güncellemesiyle Hazır!")

[BAŞARILI] İstemci Altyapısı Dinamik LR Güncellemesiyle Hazır!


hücre 4 sözde kod

In [7]:
# ==============================================================================
# HÜCRE 5: SUNUCU, EARLY STOPPING VE KAPSAMLI METRİK MOTORU
# ==============================================================================
import os
import time
import flwr as fl
import torch
import torch.nn as nn
import numpy as np
from collections import OrderedDict
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             cohen_kappa_score, roc_auc_score, average_precision_score,
                             mean_absolute_error, mean_squared_error, confusion_matrix)

class EarlyStoppingException(Exception): pass

# CSV İçin Genişletilmiş Hafıza Sözlüğü
global_history = {
    "Epoch": [], "Train_Loss": [], "Val_Loss": [], "Accuracy": [], "Precision": [],
    "Recall_Sensitivity": [], "Specificity": [], "F1_Measure": [], "Cohen_Kappa": [],
    "ROC_AUC": [], "PR_AUC_uAP": [], "MAE": [], "RMSE": [],
    "Inference_Time_ms": [], "Epoch_Suresi_sn": [], "Learning_Rate": []
}

global_model = jenerik_model_olustur(CONFIG["model_architecture"]).to(device)

KAYIT_KLASORU = f"/content/drive/MyDrive/Doktora/Verisetleri_tik/deneyler/{CONFIG['experiment_name']}_Sonuclar"
os.makedirs(KAYIT_KLASORU, exist_ok=True)
MODEL_KAYIT_YOLU = os.path.join(KAYIT_KLASORU, f"best_global_model_{CONFIG['model_architecture']}.pth")

def get_evaluate_fn(model, valloader, device):
    best_val_loss = float('inf')
    patience_counter = 0

    def evaluate(server_round, parameters, config):
        nonlocal best_val_loss, patience_counter
        round_start_time = time.time()

        params_dict = zip(model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        model.load_state_dict(state_dict, strict=True)

        criterion = nn.CrossEntropyLoss()
        model.eval()
        val_loss = 0.0
        y_true, y_pred, y_prob = [], [], []

        inference_start = time.time()
        with torch.no_grad():
            for images, labels in valloader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                val_loss += criterion(outputs, labels).item()

                probs = torch.softmax(outputs, dim=1)[:, 1] # Pozitif sınıf (Kırık) olasılıkları
                _, predicted = torch.max(outputs.data, 1)

                y_true.extend(labels.cpu().numpy())
                y_pred.extend(predicted.cpu().numpy())
                y_prob.extend(probs.cpu().numpy())

        inference_end = time.time()
        inference_time_ms = ((inference_end - inference_start) / len(valloader.dataset)) * 1000
        avg_val_loss = val_loss / len(valloader)

        # Kompleks Metrik Hesaplamaları
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        kappa = cohen_kappa_score(y_true, y_pred)

        # Benzersiz sınıflar sadece 1 veya sadece 0 ise ROC_AUC hesaplanamaz, hata vermesin diye try-except
        try: roc_auc = roc_auc_score(y_true, y_prob)
        except: roc_auc = 0.0

        pr_auc = average_precision_score(y_true, y_prob)
        mae = mean_absolute_error(y_true, y_prob)
        rmse = np.sqrt(mean_squared_error(y_true, y_prob))

        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel() if len(cm.ravel()) == 4 else (0,0,0,0)
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        round_end_time = time.time()
        epoch_suresi = round_end_time - round_start_time
        # Round 0 (Başlangıç testinde) üssün negatif olmasını engelleyen güvenlik kilidi
        if server_round == 0:
            current_lr = CONFIG["learning_rate"]
        else:
            current_lr = CONFIG["learning_rate"] * (CONFIG["lr_decay"] ** (server_round - 1))

        # En Düşük Val_Loss (Merkezi Eğitim Standardı)
        durum = ""
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            durum = "🌟 YENİ EN İYİ VAL_LOSS -> KAYDEDİLDİ"
            torch.save(model.state_dict(), MODEL_KAYIT_YOLU)
        else:
            patience_counter += 1
            durum = f"⚠️ Gelişme Yok ({patience_counter}/{CONFIG['patience']})"

        print(f"[Round {server_round}] Val Loss: {avg_val_loss:.4f} | Kappa: {kappa:.4f} | {durum}")

        # CSV İçin Verileri Hafızaya Yaz (Federe öğrenmede global train_loss hesaplanmadığı için NaN bırakılır)
        global_history["Epoch"].append(server_round)
        global_history["Train_Loss"].append(np.nan)
        global_history["Val_Loss"].append(avg_val_loss)
        global_history["Accuracy"].append(acc)
        global_history["Precision"].append(prec)
        global_history["Recall_Sensitivity"].append(rec)
        global_history["Specificity"].append(specificity)
        global_history["F1_Measure"].append(f1)
        global_history["Cohen_Kappa"].append(kappa)
        global_history["ROC_AUC"].append(roc_auc)
        global_history["PR_AUC_uAP"].append(pr_auc)
        global_history["MAE"].append(mae)
        global_history["RMSE"].append(rmse)
        global_history["Inference_Time_ms"].append(inference_time_ms)
        global_history["Epoch_Suresi_sn"].append(epoch_suresi)
        global_history["Learning_Rate"].append(current_lr)

        if patience_counter >= CONFIG["patience"]:
            raise EarlyStoppingException()

        return avg_val_loss, {"kappa": kappa}
    return evaluate

def get_on_fit_config_fn():
    def fit_config(server_round: int):
        power = max(0, server_round - 1) # Negatif üs alınmasını engeller
        return {"lr": CONFIG["learning_rate"] * (CONFIG["lr_decay"] ** power), "server_round": server_round}
    return fit_config

def client_fn(cid: str) -> fl.client.Client:
    trainloader = get_client_dataloader(int(cid))
    client_model = jenerik_model_olustur(CONFIG["model_architecture"]).to(device)
    return MuraClient(cid, client_model, trainloader, trainloader, device)

strategy = fl.server.strategy.FedAvg(
    fraction_fit=CONFIG["fraction_fit"],
    min_fit_clients=int(CONFIG["num_clients"] * CONFIG["fraction_fit"]),
    min_available_clients=CONFIG["num_clients"],
    on_fit_config_fn=get_on_fit_config_fn(),
    evaluate_fn=get_evaluate_fn(global_model, global_val_loader, device),
    initial_parameters=fl.common.ndarrays_to_parameters([val.cpu().numpy() for _, val in global_model.state_dict().items()])
)

print(f"\n🚀 SİMÜLASYON BAŞLIYOR (Val_Loss Takibi Aktif) 🚀")
try:
    history = fl.simulation.start_simulation(client_fn=client_fn, num_clients=CONFIG["num_clients"], config=fl.server.ServerConfig(num_rounds=CONFIG["num_rounds"]), strategy=strategy, client_resources={"num_cpus": 2, "num_gpus": 1 if torch.cuda.is_available() else 0})
except EarlyStoppingException:
    print("\n✅ Simülasyon Erken Durdurma ile tamamlandı.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/22.4M [00:00<?, ?B/s]

INFO flwr 2026-03-30 16:22:47,374 | app.py:146 | Starting Flower simulation, config: ServerConfig(num_rounds=10, round_timeout=None)
INFO:flwr:Starting Flower simulation, config: ServerConfig(num_rounds=10, round_timeout=None)



🚀 SİMÜLASYON BAŞLIYOR (Val_Loss Takibi Aktif) 🚀


2026-03-30 16:22:54,120	INFO worker.py:2013 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
INFO flwr 2026-03-30 16:22:58,456 | app.py:180 | Flower VCE: Ray initialized with resources: {'accelerator_type:A100': 1.0, 'node:172.28.0.12': 1.0, 'memory': 125178247578.0, 'GPU': 1.0, 'CPU': 12.0, 'node:__internal_head__': 1.0, 'object_store_memory': 53647820390.0}
INFO:flwr:Flower VCE: Ray initialized with resources: {'accelerator_type:A100': 1.0, 'node:172.28.0.12': 1.0, 'memory': 125178247578.0, 'GPU': 1.0, 'CPU': 12.0, 'node:__internal_head__': 1.0, 'object_store_memory': 53647820390.0}
INFO flwr 2026-03-30 16:22:58,458 | server.py:86 | Initializing global parameters
INFO:

...

DEBUG flwr 2026-03-30 17:13:39,768 | server.py:182 | evaluate_round 8 received 0 results and 5 failures
DEBUG:flwr:evaluate_round 8 received 0 results and 5 failures
DEBUG flwr 2026-03-30 17:13:39,769 | server.py:218 | fit_round 9: strategy sampled 5 clients (out of 5)
DEBUG:flwr:fit_round 9: strategy sampled 5 clients (out of 5)
DEBUG flwr 2026-03-30 17:19:54,299 | server.py:232 | fit_round 9 received 5 results and 0 failures
DEBUG:flwr:fit_round 9 received 5 results and 0 failures
INFO flwr 2026-03-30 17:19:57,597 | server.py:119 | fit progress: (9, 0.4765242351591587, {'kappa': np.float64(0.5856027667182445)}, 3414.443360151)
INFO:flwr:fit progress: (9, 0.4765242351591587, {'kappa': np.float64(0.5856027667182445)}, 3414.443360151)
DEBUG flwr 2026-03-30 17:19:57,598 | server.py:168 | evaluate_round 9: strategy sampled 5 clients (out of 5)
DEBUG:flwr:evaluate_round 9: strategy sampled 5 clients (out of 5)


[Round 9] Val Loss: 0.4765 | Kappa: 0.5856 | ⚠️ Gelişme Yok (1/12)


DEBUG flwr 2026-03-30 17:19:59,054 | server.py:182 | evaluate_round 9 received 0 results and 5 failures
DEBUG:flwr:evaluate_round 9 received 0 results and 5 failures
DEBUG flwr 2026-03-30 17:19:59,055 | server.py:218 | fit_round 10: strategy sampled 5 clients (out of 5)
DEBUG:flwr:fit_round 10: strategy sampled 5 clients (out of 5)
DEBUG flwr 2026-03-30 17:26:13,246 | server.py:232 | fit_round 10 received 5 results and 0 failures
DEBUG:flwr:fit_round 10 received 5 results and 0 failures
INFO flwr 2026-03-30 17:26:16,527 | server.py:119 | fit progress: (10, 0.4768272347003222, {'kappa': np.float64(0.5869800776603067)}, 3793.3733413669997)
INFO:flwr:fit progress: (10, 0.4768272347003222, {'kappa': np.float64(0.5869800776603067)}, 3793.3733413669997)
DEBUG flwr 2026-03-30 17:26:16,528 | server.py:168 | evaluate_round 10: strategy sampled 5 clients (out of 5)
DEBUG:flwr:evaluate_round 10: strategy sampled 5 clients (out of 5)


[Round 10] Val Loss: 0.4768 | Kappa: 0.5870 | ⚠️ Gelişme Yok (2/12)


DEBUG flwr 2026-03-30 17:26:18,019 | server.py:182 | evaluate_round 10 received 0 results and 5 failures
DEBUG:flwr:evaluate_round 10 received 0 results and 5 failures
INFO flwr 2026-03-30 17:26:18,021 | server.py:147 | FL finished in 3794.867365594
INFO:flwr:FL finished in 3794.867365594
INFO flwr 2026-03-30 17:26:18,024 | app.py:218 | app_fit: losses_distributed []
INFO:flwr:app_fit: losses_distributed []
INFO flwr 2026-03-30 17:26:18,025 | app.py:219 | app_fit: metrics_distributed_fit {}
INFO:flwr:app_fit: metrics_distributed_fit {}
INFO flwr 2026-03-30 17:26:18,027 | app.py:220 | app_fit: metrics_distributed {}
INFO:flwr:app_fit: metrics_distributed {}
INFO flwr 2026-03-30 17:26:18,028 | app.py:221 | app_fit: losses_centralized [(0, 12.593270037174225), (1, 0.5504524044692516), (2, 0.5161212834715844), (3, 0.5023807859420777), (4, 0.4855010211467743), (5, 0.48354405775666237), (6, 0.48365717232227323), (7, 0.483621843457222), (8, 0.47497825279831885), (9, 0.4765242351591587), (10, 

hücre 5 sözde kod

hücre 6 sözde kod

In [8]:
# ==============================================================================
# HÜCRE 6: ÇIKTILARI, GRAFİKLERİ VE HİPERPARAMETRELERİ KAYDETME BÖLÜMÜ
# ==============================================================================
import os
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve

print("\nSonuçlar, grafikler ve hiperparametreler Google Drive'a kaydediliyor...")

# Drive üzerinde bu deneye özel benzersiz sonuç klasörü (Hücre 5'ten gelen KAYIT_KLASORU)
grafik_klasoru = KAYIT_KLASORU
os.makedirs(grafik_klasoru, exist_ok=True)

# 1. Epok epok tüm eğitim geçmişi sayısal formatta (CSV) kaydedilir
df_sonuclar = pd.DataFrame(global_history)
csv_yolu = os.path.join(grafik_klasoru, f"{CONFIG['model_architecture']}_Egitim_Metrikleri.csv")
df_sonuclar.to_csv(csv_yolu, index=False)

# 2. Deneyin birebir tekrar üretilebilmesi için CONFIG ayarları JSON olarak kaydedilir
toplam_sure_dk = sum(global_history["Epoch_Suresi_sn"]) / 60.0

kayit_verisi = CONFIG.copy()
kayit_verisi["Zaman_Bilgileri"] = {
    "Toplam_Egitim_Suresi_Dakika": round(toplam_sure_dk, 2),
    "Epoch_Sureleri_Saniye": [round(sn, 2) for sn in global_history["Epoch_Suresi_sn"]]
}
kayit_verisi["Learning_Rate_Gecmisi"] = global_history["Learning_Rate"]

hiperparametre_dosyasi = os.path.join(grafik_klasoru, f"{CONFIG['model_architecture']}_Hiperparametreler.json")
with open(hiperparametre_dosyasi, "w", encoding="utf-8") as f:
    json.dump(kayit_verisi, f, indent=4, ensure_ascii=False)

# --- MATPLOTLIB İLE GÖRSELLEŞTİRME (KAYIP VE DOĞRULUK EĞRİLERİ) ---
# 1. Eğitim ve Doğrulama Kaybı (Loss) Eğrisi
plt.figure(figsize=(10, 5))
# FL'de Train_Loss NaN olduğu için çizgi görünmeyecek ancak grafiğin formatı korunacaktır.
plt.plot(df_sonuclar['Epoch'], df_sonuclar['Train_Loss'], label='Training Loss', marker='o', markersize=4)
plt.plot(df_sonuclar['Epoch'], df_sonuclar['Val_Loss'], label='Validation Loss', marker='o', markersize=4)
plt.title('Training and Validation Loss')
plt.xlabel('Epoch (Global Round)')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(grafik_klasoru, "1_Training_Validation_Loss_Curve.png"), dpi=300)
plt.close()

# 2. Accuracy Eğrisi
plt.figure(figsize=(10, 5))
plt.plot(df_sonuclar['Epoch'], df_sonuclar['Accuracy'], label='Accuracy Curve', color='green', marker='o', markersize=4)
plt.title('Accuracy')
plt.xlabel('Epoch (Global Round)')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(grafik_klasoru, "2_Accuracy_Curve.png"), dpi=300)
plt.close()

# ==============================================================================
# KRİTİK DÜZELTME: EN İYİ MODELİ GERİ YÜKLEME VE YENİDEN TAHMİN (BEST MODEL INFERENCE)
# ==============================================================================
print("\nGrafikler için 'En İyi Model (Best Model)' ağırlıkları disken geri yükleniyor...")
best_model = jenerik_model_olustur(CONFIG["model_architecture"]).to(device)
best_model.load_state_dict(torch.load(MODEL_KAYIT_YOLU))
best_model.eval()

y_true_best, y_pred_best, y_scores_best = [], [], []

# Sadece doğrulama seti üzerinden en iyi ağırlıklarla tekrar tahmin (inference) alıyoruz
with torch.no_grad():
    for inputs, labels in tqdm(global_val_loader, desc="Best Model Değerlendirmesi"):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = best_model(inputs)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

        y_true_best.extend(labels.cpu().numpy())
        y_pred_best.extend(preds.cpu().numpy())
        y_scores_best.extend(probs[:, 1].cpu().numpy())

# --- NİHAİ BAŞARIM GRAFİKLERİ (BEST MODEL ÜZERİNDEN) ---
# 3. Karmaşıklık Matrisi (Heatmap)
cm_default = confusion_matrix(y_true_best, y_pred_best)
tn, fp, fn, tp = cm_default.ravel() if cm_default.size == 4 else (0, 0, 0, 0)
cm_custom = np.array([[tp, fp],
                      [fn, tn]])
plt.figure(figsize=(8, 6))
sns.heatmap(cm_custom, annot=True, fmt='d', cmap='Wistia', cbar=False,
            annot_kws={"size": 16, "weight": "bold"},
            xticklabels=['Actual Positive (1)', 'Actual Negative (0)'],
            yticklabels=['Predicted Positive (1)', 'Predicted Negative (0)'],
            linewidths=1, linecolor='black')
plt.title('Confusion Matrix', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Actual Values', fontsize=14, fontweight='bold', labelpad=10)
plt.ylabel('Predicted Values', fontsize=14, fontweight='bold', labelpad=10)
plt.yticks(rotation=90, va="center")
plt.tight_layout()
plt.savefig(os.path.join(grafik_klasoru, "3_Confusion_Matrix.png"), dpi=300)
plt.close()

# 4. ROC Eğrisi
fpr, tpr, _ = roc_curve(y_true_best, y_scores_best)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(grafik_klasoru, "4_ROC_Curve.png"), dpi=300)
plt.close()

# 5. Kesinlik-Duyarlılık (Precision-Recall) Eğrisi
precision, recall, _ = precision_recall_curve(y_true_best, y_scores_best)
plt.figure(figsize=(7, 6))
plt.plot(recall, precision, color='purple', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(grafik_klasoru, "5_PR_Curve.png"), dpi=300)
plt.close()

print(f"\n✅ Tüm işlemler başarılı! Metrikler ve {CONFIG['experiment_name']} grafikleri '{grafik_klasoru}' klasörüne kaydedildi.")


Sonuçlar, grafikler ve hiperparametreler Google Drive'a kaydediliyor...

Grafikler için 'En İyi Model (Best Model)' ağırlıkları disken geri yükleniyor...


Best Model Değerlendirmesi: 100%|██████████| 100/100 [00:03<00:00, 33.13it/s]



✅ Tüm işlemler başarılı! Metrikler ve Exp 6.1.1.2: MobileViT-S_FL_IID_LocalEpoch:5_Round:10 grafikleri '/content/drive/MyDrive/Doktora/Verisetleri_tik/deneyler/Exp 6.1.1.2: MobileViT-S_FL_IID_LocalEpoch:5_Round:10_Sonuclar' klasörüne kaydedildi.


In [9]:
from IPython.display import Audio, display

display(Audio(url="https://www.soundjay.com/door_c2026/sounds/doorbell-7.mp3", autoplay=True))
